In [ ]:
import sys, os, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install",
    "flask", "flask-cors", "pyngrok", "openai", "requests", "-q"])
for url in [
    "https://raw.githubusercontent.com/omar-florez/agent-coliseum/bcp/agent_base.py",
    "https://raw.githubusercontent.com/omar-florez/agent-coliseum/bcp/agent_server.py",
]:
    subprocess.check_call(["wget", "-q", "-N", url])
os.environ["AZURE_API_KEY"]  = "YOUR_AZURE_API_KEY_HERE"
os.environ["AZURE_BASE_URL"] = "https://rsgd15-foundry.openai.azure.com/openai/v1/"
os.environ["NGROK_TOKEN"]    = "YOUR_NGROK_TOKEN_HERE"
print(f"Python: {sys.executable}")
print("Setup complete")

In [ ]:
# ============================================================
#  AGENT COLISEUM — BCP Branch — Colab 03: Naive Baseline
# ============================================================
#  Strategy: Simplest possible agent — no memory, no RAG.
#  Used during the talk to contrast with smarter agents.
#  Shows what agency buys you when comparing to Colabs 01 and 02.
#
#  TALK NARRATIVE — why this agent loses:
#    Turn 1: No history awareness — may repeat topics
#    Turn 3: No memory — contradicts itself
#    Turn 5: No RAG — misses obscure LatAm facts
#    Same LLM, same Azure key, radically different performance.
# ============================================================

import os, random
from openai import OpenAI
from agent_base import Agent, MatchContext, MatchResult, WorldContext, Position
from agent_server import serve_and_register

# ── Config ────────────────────────────────────────────────────────────────
AZURE_API_KEY  = os.environ.get("AZURE_API_KEY",  "YOUR_AZURE_API_KEY_HERE")
AZURE_BASE_URL = os.environ.get("AZURE_BASE_URL", "https://rsgd15-foundry.openai.azure.com/openai/v1/")
MODEL          = "gpt-5"
ARENA_URL      = "https://agent-coliseum.onrender.com"
NGROK_TOKEN    = os.environ.get("NGROK_TOKEN", "YOUR_NGROK_TOKEN_HERE")

client = OpenAI(base_url=AZURE_BASE_URL, api_key=AZURE_API_KEY)
print(f"Client ready: {client.base_url}")

# ── Agent ─────────────────────────────────────────────────────────────────

class NaiveAgent(Agent):
    """
    Naive baseline — no memory, no RAG, no strategy.
    Same LLM as the other agents, radically simpler prompting.
    Intentionally weak for demo contrast purposes.
    """

    name        = "Bot Basico"
    avatar      = "🤖"
    description = "Un agente simple sin memoria ni recuperacion de conocimiento"

    def think(self, ctx: MatchContext) -> str:
        """
        Minimal one-shot prompt with 6-step structure.
        No history, no memory, no RAG — shows the baseline.
        """
        if ctx.role == "asker":
            prompt = f"""You are competing in a Latin America knowledge contest.
Topic: {ctx.topic}
Role: asker

# SITUATION  [Chain-of-Thought — Wei et al. 2022, NeurIPS]
SITUATION: <current match state>

# OPPONENT  [Theory of Mind — Langner et al. 2023]
OPPONENT: <what you know about them>

# GOAL  [ReAct — Yao et al. 2022, ICLR 2023]
GOAL: <objective this turn>

# DRAFT  [Scratchpad — Nye et al. 2021]
DRAFT: <first question attempt>

# CRITIQUE  [Self-Refine — Madaan et al. 2023, NeurIPS]
CRITIQUE: <is it good enough?>

# FINAL  [Reflexion — Shinn et al. 2023]
FINAL: <final question (1 sentence)>"""
        else:
            prompt = f"""You are competing in a Latin America knowledge contest.
Topic: {ctx.topic}
Question: {ctx.current_question}

# SITUATION  [Chain-of-Thought — Wei et al. 2022, NeurIPS]
SITUATION: <current match state>

# OPPONENT  [Theory of Mind — Langner et al. 2023]
OPPONENT: <what you know about them>

# GOAL  [ReAct — Yao et al. 2022, ICLR 2023]
GOAL: <objective this turn>

# DRAFT  [Scratchpad — Nye et al. 2021]
DRAFT: <first answer attempt>

# CRITIQUE  [Self-Refine — Madaan et al. 2023, NeurIPS]
CRITIQUE: <is it accurate?>

# FINAL  [Reflexion — Shinn et al. 2023]
FINAL: <final answer (1-2 sentences max, be concise)>"""

        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_completion_tokens=250,
        )
        return response.choices[0].message.content.strip()

    def ask(self, ctx: MatchContext) -> str:
        return self._extract_final(self.think(ctx))

    def answer(self, ctx: MatchContext) -> str:
        return self._extract_final(self.think(ctx))

    def _extract_final(self, text: str) -> str:
        """
        Extract the FINAL answer from the reasoning trace.
        Robust to GPT formatting: "FINAL: ...", "**FINAL:**", FINAL on own line.
        Falls back to last non-empty non-comment line.
        """
        lines = text.split("\n")
        for i, line in enumerate(lines):
            clean = line.replace("**", "").strip()
            if clean.upper().startswith("FINAL:"):
                rest = clean[6:].strip()
                if rest:
                    return rest
                for j in range(i+1, len(lines)):
                    if lines[j].strip():
                        return lines[j].strip()
            elif clean.upper() == "FINAL":
                for j in range(i+1, len(lines)):
                    if lines[j].strip():
                        return lines[j].strip()
        for line in reversed(lines):
            if line.strip() and not line.strip().startswith("#"):
                return line.strip()
        return text.strip()

# ── Keepalive ─────────────────────────────────────────────────────────────
import threading, requests as _req, time as _time

def _keepalive(arena_url):
    """Ping arena every 60s to keep ngrok tunnel and Colab session alive."""
    while True:
        _time.sleep(60)
        try:
            _req.get(f"{arena_url}/health", timeout=5)
        except:
            pass

threading.Thread(target=_keepalive, args=(ARENA_URL,), daemon=True).start()
print("Keepalive thread started")

# ── Run ───────────────────────────────────────────────────────────────────
agent = NaiveAgent()

In [ ]:
# ── DEBUG CELL — run this to inspect raw GPT-5 output ────────────────────
# This helps verify that _extract_final works correctly.
# Run this cell after the agent is defined, before serve_and_register.

from agent_base import MatchContext

test_ctx = MatchContext(
    match_id="test", topic="Latin American geography",
    turn=1, total_turns=3, role="asker",
    history=[], my_agent_id="me", opponent_agent_id="opp",
    opponent_name="Test Opponent", my_scores=[], opponent_scores=[],
)

print("=== RAW think() OUTPUT ===")
raw = agent.think(test_ctx)
print(raw)
print()
print("=== EXTRACTED FINAL ===")
print(repr(agent._extract_final(raw)))
print()

test_ctx_answerer = MatchContext(
    match_id="test", topic="Latin American geography",
    turn=1, total_turns=3, role="answerer",
    history=[], my_agent_id="me", opponent_agent_id="opp",
    opponent_name="Test Opponent", my_scores=[], opponent_scores=[],
    current_question="What is the longest river in South America?",
)

print("=== RAW think() OUTPUT (answerer) ===")
raw2 = agent.think(test_ctx_answerer)
print(raw2)
print()
print("=== EXTRACTED FINAL ===")
print(repr(agent._extract_final(raw2)))

In [ ]:
# ── Run cell — starts the agent server ───────────────────────────────────
from pyngrok import ngrok
ngrok.kill()  # kill any existing tunnels before starting

serve_and_register(
    agent       = agent,
    arena_url   = ARENA_URL,
    port        = 5000,
    ngrok_token = NGROK_TOKEN,
)
# This cell blocks. Agent is now live and registered.
# Wait for the organizer to accept you in the admin panel.